# Tokenizer Analysis

Analyze the custom BPE tokenizer trained on our curated data mix.

- Vocab coverage and token fertility vs GPT-2 / LLaMA
- Special token round-trip validation
- Code vs prose tokenization comparison
- Unknown token rate analysis

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sentencepiece as spm

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

TOKENIZER_PATH = "/data/tokenizer/slm_tokenizer.model"

assert Path(TOKENIZER_PATH).exists(), f"Tokenizer not found: {TOKENIZER_PATH}"

sp = spm.SentencePieceProcessor()
sp.Load(TOKENIZER_PATH)

print(f"Vocab size:    {sp.GetPieceSize():,}")
print(f"BOS id:        {sp.bos_id()}")
print(f"EOS id:        {sp.eos_id()}")
print(f"PAD id:        {sp.pad_id()}")
print(f"UNK id:        {sp.unk_id()}")

## Special Token Validation

In [ ]:
special_tokens = [
    "<PAD>", "<UNK>", "<BOS>", "<EOS>",
    "<|system|>", "<|user|>", "<|assistant|>",
    "<|endofturn|>", "<|code|>", "<|endofcode|>",
]

print(f"{'Token':<20} {'ID':>6} {'Round-trip':>12}")
print("-" * 42)
all_ok = True
for tok in special_tokens:
    tok_id = sp.PieceToId(tok)
    decoded = sp.IdToPiece(tok_id)
    ok = decoded == tok
    if not ok:
        all_ok = False
    status = "✓" if ok else "✗ FAIL"
    print(f"  {tok:<20} {tok_id:>6}   {status}")

print(f"\nAll special tokens valid: {'✓ YES' if all_ok else '✗ NO'}")

## Fertility: Tokens per Word

Lower fertility = more efficient tokenization for that domain.

In [ ]:
# Sample texts from different domains
samples = {
    "General English": [
        "The quick brown fox jumps over the lazy dog.",
        "Machine learning is a subset of artificial intelligence that enables systems to learn from data.",
        "Python is a high-level, interpreted programming language with dynamic semantics.",
    ],
    "Python code": [
        "def fibonacci(n: int) -> int:\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
        "import numpy as np\narr = np.array([1, 2, 3, 4, 5])\nprint(arr.mean(), arr.std())",
        "class DataLoader:\n    def __init__(self, dataset, batch_size=32):\n        self.dataset = dataset\n        self.batch_size = batch_size",
    ],
    "Technical text": [
        "The transformer architecture uses self-attention mechanisms to process sequential data in parallel.",
        "Gradient descent optimizes the loss function by iteratively moving in the direction of steepest descent.",
        "The BERT model is pre-trained on masked language modeling and next sentence prediction objectives.",
    ],
}

def fertility(text: str, tokenizer) -> float:
    words = text.split()
    tokens = tokenizer.EncodeAsIds(text)
    return len(tokens) / max(1, len(words))

print(f"{'Domain':<20} {'Avg fertility':>15} {'Tokens/word'}")
print("-" * 50)
domain_fertilities = {}
for domain, texts in samples.items():
    fertilities = [fertility(t, sp) for t in texts]
    avg = np.mean(fertilities)
    domain_fertilities[domain] = avg
    print(f"  {domain:<20} {avg:>15.2f}")

In [ ]:
# Compare with GPT-2 tokenizer if available
try:
    from transformers import GPT2Tokenizer
    gpt2 = GPT2Tokenizer.from_pretrained("gpt2")

    def gpt2_fertility(text):
        words = text.split()
        tokens = gpt2.encode(text)
        return len(tokens) / max(1, len(words))

    gpt2_fertilities = {}
    for domain, texts in samples.items():
        gpt2_fertilities[domain] = np.mean([gpt2_fertility(t) for t in texts])

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(samples))
    w = 0.35
    b1 = ax.bar(x - w/2, [domain_fertilities[d] for d in samples],
                w, label="SLM (custom)", color="#5DCAA5", edgecolor="white")
    b2 = ax.bar(x + w/2, [gpt2_fertilities[d] for d in samples],
                w, label="GPT-2", color="#85B7EB", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(list(samples.keys()))
    ax.set_ylabel("Fertility (tokens/word)")
    ax.set_title("Tokenizer fertility comparison — SLM custom vs GPT-2")
    ax.legend()
    ax.set_ylim(0, max(max(domain_fertilities.values()), max(gpt2_fertilities.values())) * 1.3)
    plt.tight_layout()
    plt.savefig("/results/eval/tokenizer_fertility.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print(f"GPT-2 tokenizer not available for comparison: {e}")
    print("Install transformers to enable comparison: pip install transformers")

## Code Tokenization Example

In [ ]:
code_sample = """def quicksort(arr: list[int]) -> list[int]:
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left  = [x for x in arr if x < pivot]
    mid   = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + mid + quicksort(right)"""

tokens = sp.EncodeAsPieces(code_sample)
token_ids = sp.EncodeAsIds(code_sample)

print(f"Code sample: {len(code_sample.split())} words → {len(tokens)} tokens")
print(f"Fertility: {len(tokens)/len(code_sample.split()):.2f} tokens/word")
print(f"\nFirst 30 tokens:")
print(" | ".join(tokens[:30]))

## Vocab Coverage — Top Tokens

In [ ]:
# Show token distribution by type
vocab = [sp.IdToPiece(i) for i in range(sp.GetPieceSize())]

special  = [v for v in vocab if v.startswith("<") and v.endswith(">")]
bytes_   = [v for v in vocab if v.startswith("<0x")]
subwords = [v for v in vocab if not v.startswith("<")]
with_space = [v for v in subwords if v.startswith("▁")]

print(f"Vocab breakdown ({sp.GetPieceSize():,} total):")
print(f"  Special tokens:     {len(special):>6,}")
print(f"  Byte fallback:      {len(bytes_):>6,}")
print(f"  Subword (w/ space): {len(with_space):>6,}")
print(f"  Subword (no space): {len(subwords)-len(with_space):>6,}")
print(f"\nSample subwords (word-starting):")
print(" | ".join(with_space[100:120]))